# L07-02｜Qwen3-0.6B 的 LoRA smoke test


## 这次只看流程

代码启动 3 个 **step（训练步）**。一个 step 处理一批数据并更新一次参数。3 步只用来检查命令、设备、数据和日志是否接通，不能说明模型已经学好。

训练采用 **SFT（监督微调）**：给模型一段对话和期望回答，让它继续学习。数据写成 JSONL，每行一条消息记录。初始化代码会在当前工作目录生成数据，不需要预先准备外部文件。


## 运行时看四处

| 环节 | 代码在做什么 | 你可以观察什么 |
| --- | --- | --- |
| 数据 | 检查首条消息记录的字段 | 数据格式是否能被读取 |
| LoRA | 构造 `swift sft` 命令 | 哪些参数控制适配器训练 |
| 训练 | 在单卡 NPU 上跑 3 步 | step 是否推进、进程是否正常退出 |
| 日志 | 查找 `logging.jsonl` 和 checkpoint | 是否留下真实 loss 与阶段性产物 |


## 初始化：依赖、NPU、模型和数据

第一个代码单元使用当前 Notebook 的 Python 解释器安装缺失的 Python 包，加载 CANN 环境，检查 NPU，并从 ModelScope 下载 `Qwen/Qwen3-0.6B`。ModelArts 镜像需要使用 Python 3.10–3.12，并带有匹配的 Ascend/CANN/`torch_npu` 运行时。训练数据也在这里生成，本文件不读取上一个 Notebook 或仓库里的环境变量。


In [ ]:
from __future__ import annotations

import importlib.util
import json
import math
import os
import shlex
import shutil
import site
import subprocess
import sys
from pathlib import Path

def require(condition: bool, message: str) -> None:
    if not condition:
        raise RuntimeError(message)

require((3, 10) <= sys.version_info[:2] <= (3, 12), '本流程需要 Python 3.10、3.11 或 3.12；请使用带匹配 Ascend 运行时的 ModelArts 镜像。')
def install_missing_packages() -> None:
    packages = {'modelscope': 'modelscope', 'swift': 'ms-swift', 'matplotlib': 'matplotlib'}
    missing = [dist for module, dist in packages.items() if importlib.util.find_spec(module) is None]
    if missing:
        print('安装缺失依赖：', missing)
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade', *missing])
    python_bin = str(Path(sys.executable).parent)
    user_bin = str(Path(site.getuserbase()) / 'bin')
    old_path = os.environ.get('PATH', '')
    path_entries = old_path.split(os.pathsep) if old_path else []
    for candidate in (python_bin, user_bin):
        if candidate not in path_entries:
            old_path = candidate + os.pathsep + old_path
            path_entries.insert(0, candidate)
    os.environ['PATH'] = old_path

install_missing_packages()

def load_ascend_env() -> None:
    candidates = [
        Path('/usr/local/Ascend/ascend-toolkit/set_env.sh'),
        Path('/usr/local/Ascend/ascend-toolkit/latest/set_env.sh'),
    ]
    ascend_root = Path('/usr/local/Ascend')
    if ascend_root.is_dir():
        candidates.extend(sorted(ascend_root.glob('**/set_env.sh')))
    seen = set()
    for script in candidates:
        if not script.is_file() or str(script) in seen:
            continue
        seen.add(str(script))
        result = subprocess.run(
            ['bash', '-lc', f'source {shlex.quote(str(script))} >/dev/null 2>&1 && env -0'],
            stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, check=False,
        )
        if result.returncode != 0:
            continue
        for item in result.stdout.split(b'\0'):
            if b'=' in item:
                key, value = item.split(b'=', 1)
                os.environ[key.decode()] = value.decode(errors='ignore')
        print('已加载 Ascend 环境：', script)
        return
    print('未找到 CANN set_env.sh；继续使用当前 ModelArts 进程环境。')

load_ascend_env()
os.environ.setdefault('ASCEND_RT_VISIBLE_DEVICES', '0')
import torch
try:
    import torch_npu  # noqa: F401
except Exception as exc:
    raise RuntimeError('当前环境无法导入 torch_npu。请使用带 Ascend/CANN 运行时的 ModelArts 镜像。') from exc
require(hasattr(torch, 'npu'), '当前 PyTorch 没有 torch.npu；请检查 ModelArts 的 Ascend 运行时。')
npu_count = torch.npu.device_count()
require(npu_count > 0, '没有检测到 NPU。请确认 ModelArts 实例规格和可见设备。')
torch_version = getattr(torch, '__version__', 'unknown')
torch_npu_version = getattr(torch_npu, '__version__', 'unknown')
def major_minor(version: str) -> tuple[int, int] | None:
    try:
        parts = version.split('+', 1)[0].split('.')
        return int(parts[0]), int(parts[1])
    except (IndexError, ValueError):
        return None
if major_minor(torch_version) and major_minor(torch_npu_version):
    require(major_minor(torch_version) == major_minor(torch_npu_version), f'torch 与 torch_npu 版本不匹配：{torch_version} vs {torch_npu_version}。请按同一套 CANN/PyTorch/torch_npu 重新准备 ModelArts 镜像。')
torch.npu.set_device(0)
probe = torch.zeros(1, device='npu:0')
del probe

NOTEBOOK_ID = 'L07-02'
WORK_DIR = Path(os.environ.get('L07_WORK_DIR', str(Path.cwd() / f'{NOTEBOOK_ID}_workspace'))).expanduser()
WORK_DIR.mkdir(parents=True, exist_ok=True)
MODEL_ID = os.environ.get('L07_MODEL_ID', 'Qwen/Qwen3-0.6B')
MODEL_CACHE = Path(os.environ.get('MODELSCOPE_CACHE', str(WORK_DIR / 'modelscope_cache'))).expanduser()
MODEL_CACHE.mkdir(parents=True, exist_ok=True)
MODEL_PATH_OVERRIDE = os.environ.get('L07_MODEL_PATH')
if MODEL_PATH_OVERRIDE:
    MODEL_PATH = Path(MODEL_PATH_OVERRIDE).expanduser()
    require(MODEL_PATH.is_dir(), f'指定的模型目录不存在：{MODEL_PATH}')
else:
    from modelscope import snapshot_download
    MODEL_PATH = Path(snapshot_download(MODEL_ID, cache_dir=str(MODEL_CACHE)))
require((MODEL_PATH / 'config.json').is_file(), f'模型目录缺少 config.json：{MODEL_PATH}')
MODEL_REF = str(MODEL_PATH)

def qwen3_answer(text: str) -> str:
    return '<think>\n\n</think>\n\n' + text

examples = [
    ('请用一句话解释 LoRA 与全参数微调的区别。', 'LoRA 只训练注入的低秩适配器参数，全参数微调会更新模型的全部参数。'),
    ('Python 中如何获取列表长度？', '使用内置函数 len，例如 len([1, 2, 3]) 的结果是 3。'),
    ('一个 batch 有 2 条样本，梯度累积 4 步，完成一次更新前处理多少条样本？', '在没有丢弃样本的情况下，会先处理 2 乘以 4，也就是 8 条样本。'),
    ('把“先检查日志，再判断原因”翻译成英文。', 'Check the logs first, then identify the cause.'),
    ('为什么训练 loss 下降不能单独证明模型效果更好？', '因为 loss 只反映训练目标，还需要独立评估集和明确的评价指标。'),
    ('请说出一次可追溯训练至少要保留的一项证据。', '可以保留原始 logging.jsonl、启动命令、配置或 checkpoint 路径。'),
    ]
records = [
    {'messages': [
        {'role': 'system', 'content': '你是一个简洁、准确的课程实验助手。'},
        {'role': 'user', 'content': question + ' /no_think'},
        {'role': 'assistant', 'content': qwen3_answer(answer)},
    ]}
    for question, answer in examples
]
TRAIN_DATA = WORK_DIR / 'train.jsonl'
with TRAIN_DATA.open('w', encoding='utf-8') as handle:
    for record in records:
        handle.write(json.dumps(record, ensure_ascii=False) + '\n')
loaded_records = [json.loads(line) for line in TRAIN_DATA.read_text(encoding='utf-8').splitlines() if line.strip()]
require(len(loaded_records) == len(records), '生成的 JSONL 条数不一致。')
require(all('messages' in item for item in loaded_records), '每条训练数据都必须包含 messages。')
environment_record = {'python': sys.version.split()[0], 'torch': getattr(torch, '__version__', 'unknown'), 'torch_npu': getattr(torch_npu, '__version__', 'unknown'), 'npu_count': npu_count, 'visible_npus': os.environ['ASCEND_RT_VISIBLE_DEVICES'], 'model_id': MODEL_ID, 'model_path': str(MODEL_PATH), 'train_data': str(TRAIN_DATA)}
(WORK_DIR / 'environment.json').write_text(json.dumps(environment_record, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

SWIFT_BIN = shutil.which('swift')
require(SWIFT_BIN is not None, '找不到 swift 命令；请确认 ms-swift 已安装且当前 Python 的 bin 目录在 PATH 中。')
VISIBLE_NPUS = os.environ['ASCEND_RT_VISIBLE_DEVICES']
print({'model_id': MODEL_ID, 'model_path': MODEL_REF, 'train_data': str(TRAIN_DATA), 'work_dir': str(WORK_DIR), 'npu_count': npu_count, 'visible_npus': VISIBLE_NPUS})

def run_streaming(command: list[str], log_path: Path, env: dict[str, str]) -> None:
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print('将执行：\n' + shlex.join(command))
    with log_path.open('w', encoding='utf-8') as stream:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env, bufsize=1)
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end='')
            stream.write(line)
        returncode = process.wait()
    require(returncode == 0, f'命令失败（退出码 {returncode}）。请检查：{log_path}')
    print('命令输出已保存到：', log_path)


In [ ]:
# 这些参数是本节观察 LoRA 训练过程的起点。
LORA_RANK = 8
LORA_ALPHA = 32
LEARNING_RATE = 1e-4
PER_DEVICE_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 1
MAX_LENGTH = 512
print('环境、模型和本地 JSONL 数据已准备好。')


### 这些参数分别管什么

`lora_rank` 控制低秩分支的维度，`lora_alpha` 控制这条分支的缩放。两者都不是学习率；真正决定每次参数更新幅度的是 `learning_rate`。


## 检查数据结构

这一步只检查第一条记录的字段和 `messages` 结构，不打印训练内容。检查通过，说明数据形状基本可读；它不能说明数据质量、授权或任务效果。


In [ ]:
def read_first_record(path: Path) -> dict:
    raw = path.read_text(encoding='utf-8').lstrip()
    require(raw, f'数据文件为空：{path}')
    if raw.startswith('['):
        records = json.loads(raw)
        require(isinstance(records, list) and records and isinstance(records[0], dict), 'JSON 数组的第一项必须是对象。')
        return records[0]
    return json.loads(next(line for line in raw.splitlines() if line.strip()))

first_record = read_first_record(TRAIN_DATA)
require(isinstance(first_record, dict), '训练数据的首条记录必须是对象。')
keys = sorted(first_record.keys())
accepted = {'messages', 'conversations', 'instruction', 'query', 'response'}
require(set(keys) & accepted, f'未识别到常用指令数据字段，当前仅看到：{keys}')
if 'messages' in first_record:
    require(isinstance(first_record['messages'], list) and first_record['messages'], 'messages 必须是非空列表。')
    require({'role', 'content'} <= set(first_record['messages'][0]), 'messages 的首项至少需要 role 与 content 字段。')
print('数据结构检查通过；仅显示字段名，不展示样本内容：', keys)


### 这里会留下什么

保留数据路径和字段检查结果即可。格式能读，不等于数据质量已经验证。


## 生成 3 步训练命令

这段训练只运行 3 步，用来确认命令、LoRA、NPU 和日志都能工作。它会保存一个小 checkpoint，下一格会检查这个产物。


In [ ]:
SMOKE_STEPS = 3
SMOKE_DIR = WORK_DIR / 'L07-02_smoke'
command = [
    SWIFT_BIN, 'sft',
    '--model', MODEL_REF,
    '--dataset', str(TRAIN_DATA),
    '--torch_dtype', 'bfloat16',
    '--tuner_type', 'lora',
    '--target_modules', 'all-linear',
    '--lora_rank', str(LORA_RANK),
    '--lora_alpha', str(LORA_ALPHA),
    '--loss_scale', 'ignore_empty_think',
    '--num_train_epochs', '1',
    '--max_steps', str(SMOKE_STEPS),
    '--per_device_train_batch_size', str(PER_DEVICE_BATCH_SIZE),
    '--gradient_accumulation_steps', str(GRADIENT_ACCUMULATION_STEPS),
    '--learning_rate', str(LEARNING_RATE),
    '--max_length', str(MAX_LENGTH),
    '--logging_steps', '1',
    '--save_steps', str(SMOKE_STEPS),
    '--save_total_limit', '1',
    '--output_dir', str(SMOKE_DIR),
]
env = os.environ.copy()
env['ASCEND_RT_VISIBLE_DEVICES'] = VISIBLE_NPUS
env['NPROC_PER_NODE'] = '1'
env['PYTHONUNBUFFERED'] = '1'


### 命令里的关键参数

- `--tuner_type lora` 只训练 LoRA 适配器，不更新基础模型的全部参数。
- `--target_modules all-linear` 请求向线性层注入 LoRA；到底注入了哪些层，要看训练输出。
- `--max_steps 3` 把验证范围固定在 3 步。
- `--logging_steps 1` 让每一步都写入日志；`--save_steps 3` 确保能留下 checkpoint。

`bfloat16` 是本次训练使用的浮点格式。这 3 步只用来检查流程，不用来比较模型效果。


## 执行 smoke test

运行后看控制台：step 是否推进，是否出现 OOM、NaN 或其他异常。代码会把完整输出写入 `notebook_stdout.log`。


In [ ]:
log_path = SMOKE_DIR / 'notebook_stdout.log'
run_streaming(command, log_path, env)


## 检查训练日志和 checkpoint

`logging.jsonl` 按行保存训练事件。代码检查其中是否有有限的 loss，并确认当前运行目录下有 checkpoint。这些就是本次 smoke test 的检查结果。


In [ ]:
logging_files = sorted(SMOKE_DIR.rglob('logging.jsonl'), key=lambda path: path.stat().st_mtime)
require(logging_files, f'没有在 {SMOKE_DIR} 找到 logging.jsonl。请检查命令输出：{log_path}')
logging_file = logging_files[-1]
run_dir = logging_file.parent
loss_records = []
for raw in logging_file.read_text(encoding='utf-8').splitlines():
    try:
        record = json.loads(raw)
    except json.JSONDecodeError:
        continue
    loss = record.get('loss', record.get('train_loss'))
    if isinstance(loss, (int, float)) and math.isfinite(float(loss)):
        loss_records.append(record)
require(loss_records, f'{logging_file} 中没有有限的 loss 记录。')
checkpoints = [path for path in run_dir.glob('checkpoint-*') if path.is_dir()]
require(checkpoints, f'没有找到 checkpoint；请检查 {run_dir} 的训练输出。')
print('运行目录：', run_dir)
print('logging.jsonl：', logging_file)
print('有限 loss 记录数：', len(loss_records))
print('checkpoint：', checkpoints[-1])
print('smoke test 通过：训练命令返回 0，日志含有限 loss，且已保存 checkpoint。')


## 跑完后回看

1. 3 个 step 能证明哪些环节跑通？还不能证明什么？
2. `lora_rank`、`lora_alpha` 和 `learning_rate` 各自控制什么？
3. 为什么不能拿一个 loss 数值判断训练效果？
